# Week 1a · V: collect the world, learn to see it

Ha & Schmidhuber (2018), step one and two. Collect rollouts from CarRacing with a random policy, then train a VAE that squeezes each 64×64 frame into 32 numbers.

Budget: rollouts take 20–30 minutes on Colab's two CPU cores; the VAE trains in about 15 minutes on a T4. Everything saves to Drive, and collection resumes where it stopped if Colab disconnects.

In [ ]:
# --- Setup: runs on Colab or locally (from the notebooks/ folder) -----------
REPO_URL = "https://github.com/vamsi1905/world-models-lab.git"   # <- edit once
USE_DRIVE = True        # keep data and checkpoints on Google Drive across disconnects

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("WM_SMOKE") == "1"      # tiny sizes, used only to test the notebook
if IN_COLAB:
    if not os.path.exists("/content/world-models-lab"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/world-models-lab"], check=True)
    os.chdir("/content/world-models-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "swig"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/world-models-lab"
    else:
        WORK = "/content/work"
else:
    sys.path.insert(0, os.path.abspath(".."))
    WORK = os.path.abspath(os.environ.get("WM_WORK", "../work"))
os.makedirs(WORK, exist_ok=True)

import numpy as np, torch, matplotlib.pyplot as plt
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| work dir:", WORK, "| smoke test:", SMOKE)

## 1 · Random rollouts
The policy is a slowly drifting random walk over (steer, gas, brake). Uniform random noise would leave the car shuddering on the start line; a drift sends it round bends and off the tarmac, which is the variety V and M need.

In [ ]:
from wmlab.carracing.rollouts import collect_random, load_rollouts
N_ROLL, MAX_STEPS = (8, 150) if SMOKE else (300, 600)
DATA = f"{WORK}/carracing_random"
collect_random(DATA, n_rollouts=N_ROLL, max_steps=MAX_STEPS, workers=os.cpu_count())
R = load_rollouts(DATA)
frames = np.concatenate([r["obs"] for r in R])
print(len(R), "rollouts,", len(frames), "frames,", f"{frames.nbytes/1e9:.1f} GB in RAM")
print("mean random-policy reward:", np.mean([r["rew"].sum() for r in R]).round(1))
fig, ax = plt.subplots(1, 6, figsize=(12, 2))
for a, i in zip(ax, np.linspace(0, len(frames) - 1, 6).astype(int)): a.imshow(frames[i]); a.axis("off")
plt.show()

## 2 · Train V
The loss is reconstruction plus KL divergence, with Ha's *free bits*: once KL drops below 0.5 nats per dimension it stops being penalised, so the latent keeps enough information to be useful.

In [ ]:
from wmlab.carracing.vae import train_vae, ConvVAE, to_tensor
VAE_CKPT = f"{WORK}/vae.pt"
if os.path.exists(VAE_CKPT):
    vae = ConvVAE().to(DEVICE); vae.load_state_dict(torch.load(VAE_CKPT, map_location=DEVICE)); hist = []
else:
    vae, hist = train_vae(frames, epochs=1 if SMOKE else 10, batch_size=64 if SMOKE else 128, device=DEVICE, log_every=500)
    torch.save(vae.state_dict(), VAE_CKPT)
if hist:
    h = np.array(hist); plt.figure(figsize=(6, 3)); plt.plot(h[:, 1], lw=.5, label="reconstruction"); plt.plot(h[:, 2], lw=.5, label="KL")
    plt.yscale("log"); plt.legend(); plt.title("VAE training"); plt.show()

## 3 · Look at it
Top row: real frames. Bottom: what V reconstructs from 32 numbers. Expect the road, grass and car to survive and fine texture to go. If the car vanishes, train longer; it is small and the loss barely notices it.

In [ ]:
idx = np.random.default_rng(0).choice(len(frames), 8, replace=False)
with torch.no_grad():
    x = to_tensor(frames[idx], DEVICE); mu, _ = vae.encode(x); rec = vae.decode(mu).permute(0, 2, 3, 1).cpu().numpy()
fig, ax = plt.subplots(2, 8, figsize=(14, 3.6))
for j in range(8):
    ax[0, j].imshow(frames[idx[j]]); ax[1, j].imshow(rec[j]); ax[0, j].axis("off"); ax[1, j].axis("off")
ax[0, 0].set_title("real", loc="left"); ax[1, 0].set_title("reconstructed from z", loc="left")
plt.savefig(f"{WORK}/vae_reconstructions.png", dpi=150, bbox_inches="tight"); plt.show()

### Walk the latent space
Vary one dimension of z while holding the rest fixed. Some dimensions will turn the road; some will do nothing. The ones that do nothing are the price of the KL term.

In [ ]:
with torch.no_grad():
    base, _ = vae.encode(to_tensor(frames[idx[:1]], DEVICE))
    fig, ax = plt.subplots(4, 7, figsize=(10, 6))
    for row, d in enumerate([0, 1, 2, 3]):
        for col, v in enumerate(np.linspace(-2.5, 2.5, 7)):
            z = base.clone(); z[0, d] = v
            ax[row, col].imshow(vae.decode(z)[0].permute(1, 2, 0).cpu()); ax[row, col].axis("off")
        ax[row, 0].set_title(f"z[{d}]", loc="left", fontsize=8)
plt.show()